# Phase 3 — Feature Engineering
## Australian Stock Market Analytics

Create defensible analytical features from the Phase 2 cleaned dataset. This notebook preserves the source columns, documents each feature, validates calculations, and saves separate Phase 3 outputs.

**Important:** This is cross-sectional company fundamentals data, not a historical time series. We will not fabricate dates or price history. Missing annual P/E values and WOW's malformed 52-week range remain unresolved rather than imputed.

## 1. Imports

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
print("Imports successful.")

Imports successful.


## 2. Find and load the cleaned Phase 2 dataset
Expected path: `data/processed/australian_stocks_cleaned.csv`. Run this notebook from your project or notebooks folder.

In [2]:
INPUT_FILENAME = "australian_stocks_cleaned.csv"
PROJECT_ROOT = None
input_file = None

for base in [Path.cwd(), *Path.cwd().parents]:
    candidate = base / "data" / "processed" / INPUT_FILENAME
    if candidate.exists():
        PROJECT_ROOT = base
        input_file = candidate
        break

if input_file is None:
    raise FileNotFoundError(
        f"Could not find {INPUT_FILENAME}. Make sure Phase 2 saved it in "
        "the project's data/processed folder."
    )

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(input_file)

print("Project root:", PROJECT_ROOT)
print("Input:", input_file)
print("Shape:", df.shape)
display(df.head())

Project root: e:\Projects\Australian Stock Market Data Science Project
Input: e:\Projects\Australian Stock Market Data Science Project\data\processed\australian_stocks_cleaned.csv
Shape: (53, 28)


,ticker,company_name,total_shares,market_cap_m,eps_2h2024,eps_1h2025,eps_ltm,eps_ap_24_25,pe_annual_report,pe_ltm,price,book_value_per_share,week52_range,duplicate_ticker_flag,annual_pe_missing,week52_low,week52_high,market_cap_m_original,eps_ltm_original,pe_ltm_original,total_shares_outlier,market_cap_m_outlier,eps_ltm_outlier,eps_ap_24_25_outlier,pe_ltm_outlier,price_outlier,book_value_per_share_outlier,week52_range_valid
0,VEA,Viva Energy Group Ltd,1625651926,"3,511.4082",-0.3000,12.3000,12.0000,4.8000,450.0000,18.0000,2.1600,1.0270,1.41 - 2.83,False,False,1.4100,2.8300,"3,511.4082",12.0000,18.0000,False,False,False,False,False,False,False,True
1,DDR,Dicker Data Ltd,180817855,"1,871.4648",23.9700,21.7900,45.7600,43.6200,237.2765,22.6180,10.3500,1.3890,7.58 - 10.94,False,False,7.5800,10.9400,"1,871.4648",45.7600,22.6180,False,False,False,False,False,False,False,True
2,FMG,Fortescue Metals Group Ltd,3078964918,"68,075.9143",50.5000,39.5500,90.0500,109.7000,201.5497,24.5530,22.1100,9.8740,13.18 - 22.11,False,False,13.1800,22.1100,"68,075.9143",90.0500,24.5530,True,True,False,False,False,False,False,True
3,RIO,RIO Tinto Ltd,371216214,"51,402.3092",315.2000,296.0000,611.2000,669.5000,206.8260,22.6554,138.4700,54.4740,100.75 - 140.58,False,False,100.7500,140.5800,"51,402.3092",611.2000,22.6554,False,True,True,True,False,True,True,True
4,APE,Eagers Automotive Ltd,282364338,"7,578.6588",34.8000,46.4000,81.2000,80.2000,334.6633,33.0542,26.8400,4.6360,10.90 - 35.64,False,False,10.9000,35.6400,"7,578.6588",81.2000,33.0542,False,False,False,False,False,False,False,True


## 3. Validate expected columns
Column names below match your actual Phase 2 dataframe.

In [3]:
required_columns = [
    "ticker", "company_name", "total_shares", "market_cap_m",
    "eps_2h2024", "eps_1h2025", "eps_ltm", "eps_ap_24_25",
    "pe_annual_report", "pe_ltm", "price", "book_value_per_share",
    "week52_range", "week52_low", "week52_high", "week52_range_valid"
]
missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

if df["ticker"].duplicated().any():
    raise ValueError("Duplicate tickers found. Recheck Phase 2 before proceeding.")

print("Schema validation passed.")
print("Rows:", len(df), "| Unique tickers:", df["ticker"].nunique())

Schema validation passed.
Rows: 53 | Unique tickers: 53


## 4. Ensure financial fields are numeric
Unexpected strings are converted to missing values and counted in the audit.

In [4]:
numeric_columns = [
    "total_shares", "market_cap_m", "eps_2h2024", "eps_1h2025",
    "eps_ltm", "eps_ap_24_25", "pe_annual_report", "pe_ltm",
    "price", "book_value_per_share", "week52_low", "week52_high"
]
conversion_audit = []
for col in numeric_columns:
    before = int(df[col].notna().sum())
    df[col] = pd.to_numeric(df[col], errors="coerce")
    after = int(df[col].notna().sum())
    conversion_audit.append({
        "column": col,
        "non_missing_before": before,
        "non_missing_after": after,
        "converted_to_missing": before - after
    })
conversion_audit_df = pd.DataFrame(conversion_audit)
display(conversion_audit_df)
display(df[numeric_columns].describe().T)

,column,non_missing_before,non_missing_after,converted_to_missing
0,total_shares,53,53,0
1,market_cap_m,53,53,0
2,eps_2h2024,53,53,0
3,eps_1h2025,53,53,0
4,eps_ltm,53,53,0
5,eps_ap_24_25,53,53,0
6,pe_annual_report,51,51,0
7,pe_ltm,53,53,0
8,price,53,53,0
9,book_value_per_share,53,53,0


,count,mean,std,min,25%,50%,75%,max
total_shares,53.0000,"989,834,970.4340","1,741,828,422.6563","68,152,321.0000","185,340,291.0000","305,377,396.0000","1,221,597,320.0000","11,543,702,836.0000"
market_cap_m,53.0000,"22,267.2389","46,043.1047",7.6707,553.0908,"3,484.5029","18,330.9496","255,604.6406"
eps_2h2024,53.0000,54.8040,99.6198,-47.3000,5.1600,23.9700,57.5000,554.8000
eps_1h2025,53.0000,70.3785,214.1224,-56.3000,4.0300,12.3000,46.4000,"1,473.4400"
eps_ltm,53.0000,125.1825,271.3879,-36.7000,11.4000,27.2000,96.5000,"1,560.0000"
eps_ap_24_25,53.0000,127.1736,272.1159,0.3900,11.4000,27.7300,98.2000,"1,560.0000"
pe_annual_report,51.0000,445.8107,"1,215.0219",9.1958,132.5130,190.4430,313.4441,"8,666.6667"
pe_ltm,53.0000,40.2082,120.5996,-88.4741,12.0890,18.6480,28.2791,866.6667
price,53.0000,22.5255,39.5430,0.0450,2.2500,7.7500,22.1100,196.3500
book_value_per_share,53.0000,8.9558,15.5633,0.0600,1.0970,3.5350,9.7660,90.6780


## 5. Company size features
The size bands are relative to this dataset's positive market capitalizations, not official market classifications.

In [5]:
source_columns = df.columns.tolist()

df["log_market_cap_m"] = np.where(
    df["market_cap_m"] > 0, np.log(df["market_cap_m"]), np.nan
)

positive_cap = df["market_cap_m"].where(df["market_cap_m"] > 0)
try:
    df["market_cap_size_band"] = pd.qcut(
        positive_cap, q=3,
        labels=["Small (sample-relative)", "Medium (sample-relative)", "Large (sample-relative)"],
        duplicates="drop"
    )
except ValueError:
    df["market_cap_size_band"] = pd.Series(pd.NA, index=df.index, dtype="object")

display(df[["ticker", "market_cap_m", "log_market_cap_m", "market_cap_size_band"]].head(12))
display(df["market_cap_size_band"].value_counts(dropna=False))

,ticker,market_cap_m,log_market_cap_m,market_cap_size_band
0,VEA,"3,511.4082",8.1638,Medium (sample-relative)
1,DDR,"1,871.4648",7.5345,Medium (sample-relative)
2,FMG,"68,075.9143",11.1284,Large (sample-relative)
3,RIO,"51,402.3092",10.8474,Large (sample-relative)
4,APE,"7,578.6588",8.9331,Medium (sample-relative)
5,AFG,611.4601,6.4158,Small (sample-relative)
6,ASH,30.9548,3.4325,Small (sample-relative)
7,ASG,857.3504,6.7538,Small (sample-relative)
8,SGM,"3,433.7232",8.1414,Medium (sample-relative)
9,ACL,550.5593,6.3109,Small (sample-relative)


market_cap_size_band
Small (sample-relative)     18
Large (sample-relative)     18
Medium (sample-relative)    17
Name: count, dtype: int64

## 6. EPS comparison features
These are differences, not automatically growth rates: the supplied periods may not be directly comparable. Percentage difference is only calculated when the baseline EPS is positive.

In [6]:
df["eps_half_year_difference"] = df["eps_1h2025"] - df["eps_2h2024"]
df["eps_ltm_vs_annual_difference"] = df["eps_ltm"] - df["eps_ap_24_25"]
df["eps_ltm_vs_annual_pct_difference"] = np.where(
    df["eps_ap_24_25"] > 0,
    (df["eps_ltm"] - df["eps_ap_24_25"]) / df["eps_ap_24_25"] * 100,
    np.nan
)

display(df[[
    "ticker", "eps_2h2024", "eps_1h2025", "eps_half_year_difference",
    "eps_ltm", "eps_ap_24_25", "eps_ltm_vs_annual_difference",
    "eps_ltm_vs_annual_pct_difference"
]].head(12))

,ticker,eps_2h2024,eps_1h2025,eps_half_year_difference,eps_ltm,eps_ap_24_25,eps_ltm_vs_annual_difference,eps_ltm_vs_annual_pct_difference
0,VEA,-0.3000,12.3000,12.6000,12.0000,4.8000,7.2000,150.0000
1,DDR,23.9700,21.7900,-2.1800,45.7600,43.6200,2.1400,4.9060
2,FMG,50.5000,39.5500,-10.9500,90.0500,109.7000,-19.6500,-17.9125
3,RIO,315.2000,296.0000,-19.2000,611.2000,669.5000,-58.3000,-8.7080
4,APE,34.8000,46.4000,11.6000,81.2000,80.2000,1.0000,1.2469
5,AFG,5.6500,9.4500,3.8000,15.1000,15.1000,0.0000,0.0000
6,ASH,1.2700,0.2400,-1.0300,1.5100,1.5100,0.0000,0.0000
7,ASG,5.1600,11.1000,5.9400,16.2600,16.2600,0.0000,0.0000
8,SGM,4.9000,5.0000,0.1000,9.9000,9.9000,0.0000,0.0000
9,ACL,5.8300,11.3200,5.4900,17.1500,17.1500,0.0000,0.0000


## 7. Valuation features
Earnings yield is only derived from positive LTM P/E. P/B is only derived when book value per share is positive. Negative EPS and P/E observations are retained.

In [7]:
df["earnings_yield_pct"] = np.where(df["pe_ltm"] > 0, 100 / df["pe_ltm"], np.nan)

df["price_to_book"] = np.where(
    df["book_value_per_share"] > 0,
    df["price"] / df["book_value_per_share"],
    np.nan
)

df["pe_ltm_minus_annual"] = df["pe_ltm"] - df["pe_annual_report"]
df["pe_ltm_vs_annual_pct_difference"] = np.where(
    df["pe_annual_report"] > 0,
    (df["pe_ltm"] - df["pe_annual_report"]) / df["pe_annual_report"] * 100,
    np.nan
)

display(df[[
    "ticker", "price", "book_value_per_share", "price_to_book",
    "pe_ltm", "earnings_yield_pct", "pe_annual_report",
    "pe_ltm_minus_annual", "pe_ltm_vs_annual_pct_difference"
]].head(12))

,ticker,price,book_value_per_share,price_to_book,pe_ltm,earnings_yield_pct,pe_annual_report,pe_ltm_minus_annual,pe_ltm_vs_annual_pct_difference
0,VEA,2.1600,1.0270,2.1032,18.0000,5.5556,450.0000,-432.0000,-96.0000
1,DDR,10.3500,1.3890,7.4514,22.6180,4.4213,237.2765,-214.6585,-90.4677
2,FMG,22.1100,9.8740,2.2392,24.5530,4.0728,201.5497,-176.9967,-87.8179
3,RIO,138.4700,54.4740,2.5419,22.6554,4.4140,206.8260,-184.1706,-89.0461
4,APE,26.8400,4.6360,5.7895,33.0542,3.0253,334.6633,-301.6092,-90.1232
5,AFG,2.2500,0.8170,2.7540,14.9007,6.7111,149.0066,-134.1060,-90.0000
6,ASH,0.2150,0.2160,0.9954,14.2384,7.0233,142.3841,-128.1457,-90.0000
7,ASG,4.2300,2.4980,1.6934,26.0148,3.8440,260.1476,-234.1328,-90.0000
8,SGM,17.7700,13.3870,1.3274,179.4949,0.5571,"1,794.9495","-1,615.4545",-90.0000
9,ACL,2.8400,0.8810,3.2236,16.5598,6.0387,165.5977,-149.0379,-90.0000


## 8. 52-week price-position features
Position is `(price - low) / (high - low)`. A position of 0 means price is at the reported low; 1 means it is at the reported high. Invalid ranges remain missing. We flag prices outside the range instead of clipping them.

In [8]:
valid_range = (
    df["week52_range_valid"].fillna(False).astype(bool)
    & df["week52_low"].notna()
    & df["week52_high"].notna()
    & (df["week52_high"] > df["week52_low"])
)
range_width = df["week52_high"] - df["week52_low"]

df["week52_range_width"] = np.where(valid_range, range_width, np.nan)
df["price_position_52w"] = np.where(
    valid_range, (df["price"] - df["week52_low"]) / range_width, np.nan
)
df["price_distance_from_52w_low_pct"] = np.where(
    valid_range & (df["week52_low"] != 0),
    (df["price"] - df["week52_low"]) / df["week52_low"] * 100, np.nan
)
df["price_distance_from_52w_high_pct"] = np.where(
    valid_range & (df["week52_high"] != 0),
    (df["week52_high"] - df["price"]) / df["week52_high"] * 100, np.nan
)

outside = pd.Series(pd.NA, index=df.index, dtype="boolean")
known_price_range = valid_range & df["price"].notna()
outside.loc[known_price_range] = (
    (df.loc[known_price_range, "price"] < df.loc[known_price_range, "week52_low"])
    | (df.loc[known_price_range, "price"] > df.loc[known_price_range, "week52_high"])
)
df["price_outside_52w_range"] = outside

print("Valid parsed ranges:", int(valid_range.sum()))
print("Invalid/unavailable ranges:", int((~valid_range).sum()))
print("Prices outside valid ranges:", int(df["price_outside_52w_range"].fillna(False).sum()))
display(df[[
    "ticker", "price", "week52_low", "week52_high", "price_position_52w",
    "price_distance_from_52w_low_pct", "price_distance_from_52w_high_pct",
    "price_outside_52w_range"
]].head(15))

Valid parsed ranges: 52
Invalid/unavailable ranges: 1
Prices outside valid ranges: 0


,ticker,price,week52_low,week52_high,price_position_52w,price_distance_from_52w_low_pct,price_distance_from_52w_high_pct,price_outside_52w_range
0,VEA,2.1600,1.4100,2.8300,0.5282,53.1915,23.6749,False
1,DDR,10.3500,7.5800,10.9400,0.8244,36.5435,5.3931,False
2,FMG,22.1100,13.1800,22.1100,1.0000,67.7542,0.0000,False
3,RIO,138.4700,100.7500,140.5800,0.9470,37.4392,1.5009,False
4,APE,26.8400,10.9000,35.6400,0.6443,146.2385,24.6914,False
5,AFG,2.2500,1.4250,2.9200,0.5518,57.8947,22.9452,False
6,ASH,0.2150,0.1280,0.2450,0.7436,67.9687,12.2449,False
7,ASG,4.2300,1.6050,4.7000,0.8481,163.5514,10.0000,False
8,SGM,17.7700,11.6300,18.1100,0.9475,52.7945,1.8774,False
9,ACL,2.8400,2.3800,3.8300,0.3172,19.3277,25.8486,False


## 9. Data-quality flags
These flags describe the observations; they do not automatically remove companies.

In [9]:
df["annual_pe_missing_flag"] = df["pe_annual_report"].isna()
df["negative_ltm_eps_flag"] = df["eps_ltm"] < 0
df["negative_ltm_pe_flag"] = df["pe_ltm"] < 0
df["nonpositive_book_value_flag"] = (
    df["book_value_per_share"].notna() & (df["book_value_per_share"] <= 0)
)
df["invalid_52w_range_flag"] = ~valid_range

quality_summary = pd.DataFrame({
    "quality_flag": [
        "annual_pe_missing_flag", "negative_ltm_eps_flag",
        "negative_ltm_pe_flag", "nonpositive_book_value_flag",
        "invalid_52w_range_flag", "price_outside_52w_range"
    ],
    "count": [
        int(df["annual_pe_missing_flag"].sum()),
        int(df["negative_ltm_eps_flag"].sum()),
        int(df["negative_ltm_pe_flag"].sum()),
        int(df["nonpositive_book_value_flag"].sum()),
        int(df["invalid_52w_range_flag"].sum()),
        int(df["price_outside_52w_range"].fillna(False).sum())
    ]
})
display(quality_summary)

,quality_flag,count
0,annual_pe_missing_flag,2
1,negative_ltm_eps_flag,1
2,negative_ltm_pe_flag,1
3,nonpositive_book_value_flag,0
4,invalid_52w_range_flag,1
5,price_outside_52w_range,0


## 10. Validate engineered dataset
Confirm row count, ticker uniqueness, source-column retention, and absence of infinite values.

In [10]:
assert len(df) == 53, f"Expected 53 rows, got {len(df)}"
assert df["ticker"].nunique() == len(df), "Ticker uniqueness failed."
assert all(col in df.columns for col in source_columns), "A source column was removed."

engineered_numeric = [
    "log_market_cap_m", "eps_half_year_difference",
    "eps_ltm_vs_annual_difference", "eps_ltm_vs_annual_pct_difference",
    "earnings_yield_pct", "price_to_book", "pe_ltm_minus_annual",
    "pe_ltm_vs_annual_pct_difference", "week52_range_width",
    "price_position_52w", "price_distance_from_52w_low_pct",
    "price_distance_from_52w_high_pct"
]
infinite_counts = {
    col: int(np.isinf(pd.to_numeric(df[col], errors="coerce")).sum())
    for col in engineered_numeric
}
print("Shape:", df.shape)
print("Unique tickers:", df["ticker"].nunique())
display(pd.Series(infinite_counts, name="infinite_count").to_frame())
assert sum(infinite_counts.values()) == 0, "Infinite values found in engineered features."
print("Validation passed.")

Shape: (53, 47)
Unique tickers: 53


,infinite_count
log_market_cap_m,0
eps_half_year_difference,0
eps_ltm_vs_annual_difference,0
eps_ltm_vs_annual_pct_difference,0
earnings_yield_pct,0
price_to_book,0
pe_ltm_minus_annual,0
pe_ltm_vs_annual_pct_difference,0
week52_range_width,0
price_position_52w,0


Validation passed.


## 11. Review the new features

In [11]:
engineered_columns = [
    "log_market_cap_m", "market_cap_size_band",
    "eps_half_year_difference", "eps_ltm_vs_annual_difference",
    "eps_ltm_vs_annual_pct_difference", "earnings_yield_pct",
    "price_to_book", "pe_ltm_minus_annual",
    "pe_ltm_vs_annual_pct_difference", "week52_range_width",
    "price_position_52w", "price_distance_from_52w_low_pct",
    "price_distance_from_52w_high_pct", "price_outside_52w_range",
    "annual_pe_missing_flag", "negative_ltm_eps_flag",
    "negative_ltm_pe_flag", "nonpositive_book_value_flag",
    "invalid_52w_range_flag"
]
display(df[["ticker", "company_name"] + engineered_columns].head(20))
display(df[engineered_columns].describe(include="all").T)

,ticker,company_name,log_market_cap_m,market_cap_size_band,eps_half_year_difference,eps_ltm_vs_annual_difference,eps_ltm_vs_annual_pct_difference,earnings_yield_pct,price_to_book,pe_ltm_minus_annual,pe_ltm_vs_annual_pct_difference,week52_range_width,price_position_52w,price_distance_from_52w_low_pct,price_distance_from_52w_high_pct,price_outside_52w_range,annual_pe_missing_flag,negative_ltm_eps_flag,negative_ltm_pe_flag,nonpositive_book_value_flag,invalid_52w_range_flag
0,VEA,Viva Energy Group Ltd,8.1638,Medium (sample-relative),12.6000,7.2000,150.0000,5.5556,2.1032,-432.0000,-96.0000,1.4200,0.5282,53.1915,23.6749,False,False,False,False,False,False
1,DDR,Dicker Data Ltd,7.5345,Medium (sample-relative),-2.1800,2.1400,4.9060,4.4213,7.4514,-214.6585,-90.4677,3.3600,0.8244,36.5435,5.3931,False,False,False,False,False,False
2,FMG,Fortescue Metals Group Ltd,11.1284,Large (sample-relative),-10.9500,-19.6500,-17.9125,4.0728,2.2392,-176.9967,-87.8179,8.9300,1.0000,67.7542,0.0000,False,False,False,False,False,False
3,RIO,RIO Tinto Ltd,10.8474,Large (sample-relative),-19.2000,-58.3000,-8.7080,4.4140,2.5419,-184.1706,-89.0461,39.8300,0.9470,37.4392,1.5009,False,False,False,False,False,False
4,APE,Eagers Automotive Ltd,8.9331,Medium (sample-relative),11.6000,1.0000,1.2469,3.0253,5.7895,-301.6092,-90.1232,24.7400,0.6443,146.2385,24.6914,False,False,False,False,False,False
5,AFG,Australian Finance Group Ltd,6.4158,Small (sample-relative),3.8000,0.0000,0.0000,6.7111,2.7540,-134.1060,-90.0000,1.4950,0.5518,57.8947,22.9452,False,False,False,False,False,False
6,ASH,Ashley Services Group,3.4325,Small (sample-relative),-1.0300,0.0000,0.0000,7.0233,0.9954,-128.1457,-90.0000,0.1170,0.7436,67.9687,12.2449,False,False,False,False,False,False
7,ASG,Autosports Group Ltd,6.7538,Small (sample-relative),5.9400,0.0000,0.0000,3.8440,1.6934,-234.1328,-90.0000,3.0950,0.8481,163.5514,10.0000,False,False,False,False,False,False
8,SGM,Sims Ltd,8.1414,Medium (sample-relative),0.1000,0.0000,0.0000,0.5571,1.3274,"-1,615.4545",-90.0000,6.4800,0.9475,52.7945,1.8774,False,False,False,False,False,False
9,ACL,Australian Clinical Labs Ltd,6.3109,Small (sample-relative),5.4900,0.0000,0.0000,6.0387,3.2236,-149.0379,-90.0000,1.4500,0.3172,19.3277,25.8486,False,False,False,False,False,False


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
log_market_cap_m,53.0000,NaN,NaN,NaN,7.9850,2.4244,2.0374,6.3155,8.1561,9.8163,12.4514
market_cap_size_band,53,3,Small (sample-relative),18,NaN,NaN,NaN,NaN,NaN,NaN,NaN
eps_half_year_difference,53.0000,NaN,NaN,NaN,15.5745,194.6628,-124.0000,-10.9500,-1.6400,2.8300,"1,386.8800"
eps_ltm_vs_annual_difference,53.0000,NaN,NaN,NaN,-1.9911,15.2541,-88.1000,0.0000,0.0000,0.0000,16.3000
eps_ltm_vs_annual_pct_difference,53.0000,NaN,NaN,NaN,0.3934,32.2432,-171.4008,0.0000,0.0000,0.0000,150.0000
earnings_yield_pct,52.0000,NaN,NaN,NaN,11.3101,20.8654,0.1154,3.4799,5.3324,7.7867,108.7451
price_to_book,53.0000,NaN,NaN,NaN,2.6108,2.2023,0.2839,1.1646,1.8114,2.9477,10.0099
pe_ltm_minus_annual,51.0000,NaN,NaN,NaN,-404.2946,"1,094.5321","-7,800.0000",-282.0997,-171.8972,-119.6768,5.5640
pe_ltm_vs_annual_pct_difference,51.0000,NaN,NaN,NaN,-88.1961,15.8484,-114.0054,-90.0000,-90.0000,-90.0000,17.5600
week52_range_width,52.0000,NaN,NaN,NaN,8.9469,15.0343,0.0540,1.3763,3.1800,9.5250,82.9000


## 12. Feature dictionary
This documents how each new feature was derived and its limitations.

In [12]:
feature_dictionary = pd.DataFrame([
    ["log_market_cap_m", "Natural log of market capitalization (millions)", "ln(market_cap_m) when positive", "Missing for nonpositive/missing market cap"],
    ["market_cap_size_band", "Relative company-size band", "Tertiles of positive market cap", "Sample-relative, not an official classification"],
    ["eps_half_year_difference", "Difference between half-year EPS values", "eps_1h2025 - eps_2h2024", "Not automatically a growth rate"],
    ["eps_ltm_vs_annual_difference", "LTM EPS minus supplied annual EPS", "eps_ltm - eps_ap_24_25", "Periods may not be directly comparable"],
    ["eps_ltm_vs_annual_pct_difference", "Relative LTM/annual EPS difference", "(LTM - annual) / annual * 100", "Only where annual EPS is positive"],
    ["earnings_yield_pct", "Reciprocal of positive LTM P/E as percent", "100 / pe_ltm", "Only where LTM P/E is positive"],
    ["price_to_book", "Price relative to book value per share", "price / book_value_per_share", "Only where book value per share is positive"],
    ["pe_ltm_minus_annual", "LTM P/E minus annual-report P/E", "pe_ltm - pe_annual_report", "Descriptive comparison across periods"],
    ["pe_ltm_vs_annual_pct_difference", "Relative difference between P/E measures", "(LTM P/E - annual P/E) / annual P/E * 100", "Only where annual-report P/E is positive"],
    ["week52_range_width", "Width of parsed 52-week range", "week52_high - week52_low", "Only for valid ranges"],
    ["price_position_52w", "Position of price within 52-week range", "(price - low) / (high - low)", "Only for valid ranges; can be outside 0-1"],
    ["price_distance_from_52w_low_pct", "Percent distance from 52-week low", "(price - low) / low * 100", "Only for valid ranges and nonzero low"],
    ["price_distance_from_52w_high_pct", "Percent distance below 52-week high", "(high - price) / high * 100", "Only for valid ranges and nonzero high"],
    ["price_outside_52w_range", "Price is outside reported range", "price < low or price > high", "Missing if price/range unavailable"],
    ["annual_pe_missing_flag", "Annual P/E is missing", "pe_annual_report is missing", "Flags; does not impute"],
    ["negative_ltm_eps_flag", "LTM EPS is negative", "eps_ltm < 0", "Negative EPS may be legitimate"],
    ["negative_ltm_pe_flag", "LTM P/E is negative", "pe_ltm < 0", "Interpret cautiously with negative earnings"],
    ["nonpositive_book_value_flag", "Book value/share is zero or negative", "book_value_per_share <= 0", "Not automatically a data error"],
    ["invalid_52w_range_flag", "52-week range is invalid/unavailable", "Range missing or high <= low", "WOW's malformed source range remains unresolved"]
], columns=["feature", "meaning", "derivation", "caveat"])
display(feature_dictionary)

,feature,meaning,derivation,caveat
0,log_market_cap_m,Natural log of market capitalization (millions),ln(market_cap_m) when positive,Missing for nonpositive/missing market cap
1,market_cap_size_band,Relative company-size band,Tertiles of positive market cap,"Sample-relative, not an official classification"
2,eps_half_year_difference,Difference between half-year EPS values,eps_1h2025 - eps_2h2024,Not automatically a growth rate
3,eps_ltm_vs_annual_difference,LTM EPS minus supplied annual EPS,eps_ltm - eps_ap_24_25,Periods may not be directly comparable
4,eps_ltm_vs_annual_pct_difference,Relative LTM/annual EPS difference,(LTM - annual) / annual * 100,Only where annual EPS is positive
5,earnings_yield_pct,Reciprocal of positive LTM P/E as percent,100 / pe_ltm,Only where LTM P/E is positive
6,price_to_book,Price relative to book value per share,price / book_value_per_share,Only where book value per share is positive
7,pe_ltm_minus_annual,LTM P/E minus annual-report P/E,pe_ltm - pe_annual_report,Descriptive comparison across periods
8,pe_ltm_vs_annual_pct_difference,Relative difference between P/E measures,(LTM P/E - annual P/E) / annual P/E * 100,Only where annual-report P/E is positive
9,week52_range_width,Width of parsed 52-week range,week52_high - week52_low,Only for valid ranges


## 13. Save Phase 3 outputs
The Phase 2 cleaned file is not overwritten.

In [13]:
engineered_output = PROCESSED_DIR / "australian_stocks_feature_engineered.csv"
dictionary_output = PROCESSED_DIR / "feature_dictionary_phase3.csv"
audit_output = PROCESSED_DIR / "feature_engineering_conversion_audit_phase3.csv"

df.to_csv(engineered_output, index=False)
feature_dictionary.to_csv(dictionary_output, index=False)
conversion_audit_df.to_csv(audit_output, index=False)

print("Saved engineered dataset:", engineered_output)
print("Saved feature dictionary:", dictionary_output)
print("Saved numeric conversion audit:", audit_output)

Saved engineered dataset: e:\Projects\Australian Stock Market Data Science Project\data\processed\australian_stocks_feature_engineered.csv
Saved feature dictionary: e:\Projects\Australian Stock Market Data Science Project\data\processed\feature_dictionary_phase3.csv
Saved numeric conversion audit: e:\Projects\Australian Stock Market Data Science Project\data\processed\feature_engineering_conversion_audit_phase3.csv


## 14. Final summary

In [14]:
summary = pd.DataFrame([{
    "input_file": str(input_file),
    "output_file": str(engineered_output),
    "rows": len(df),
    "unique_tickers": df["ticker"].nunique(),
    "source_column_count": len(source_columns),
    "final_column_count": df.shape[1],
    "engineered_feature_count": len(engineered_columns),
    "missing_annual_pe": int(df["annual_pe_missing_flag"].sum()),
    "invalid_52w_ranges": int(df["invalid_52w_range_flag"].sum()),
    "negative_ltm_eps": int(df["negative_ltm_eps_flag"].sum()),
    "negative_ltm_pe": int(df["negative_ltm_pe_flag"].sum()),
    "prices_outside_valid_52w_ranges": int(df["price_outside_52w_range"].fillna(False).sum())
}])
display(summary)
print("PHASE 3 FEATURE ENGINEERING COMPLETED.")
print("Review the feature dictionary and quality flags before Phase 4 (EDA).")

,input_file,output_file,rows,unique_tickers,source_column_count,final_column_count,engineered_feature_count,missing_annual_pe,invalid_52w_ranges,negative_ltm_eps,negative_ltm_pe,prices_outside_valid_52w_ranges
0,e:\Projects\Australian Stock Market Data Scien...,e:\Projects\Australian Stock Market Data Scien...,53,53,28,47,19,2,1,1,1,0


PHASE 3 FEATURE ENGINEERING COMPLETED.
Review the feature dictionary and quality flags before Phase 4 (EDA).
